# Definitive Experiment — v3

## What's new vs previous runs:
1. **Lesion-ID aware split** — prevents same lesion appearing in train AND test (fixes HAM10000 duplicate image issue)
2. **5 models** — EfficientNetB0 (4M), ViT-Ti/16 (5.7M), ViT-S/16 (22M), ViT-B/16 (86M), EfficientFormer-L1 (11M)
3. **Per-class robustness for ALL 4 degradation types** (not just noise)
4. **Same pretraining dataset** — all ViT variants use ImageNet-1k weights for fair comparison
5. **Google Drive saving** after every model
6. **Resume logic** — safely reconnect and continue

## Runtime estimate:
- 5 models × 25 epochs × 3 seeds = 15 training runs
- ~5-6 hours per seed on T4 GPU
- Run one seed per day across 3 days

---
**Set your seed before running.**

In [ ]:
# ── SET SEED BEFORE RUNNING ─────────────────────────────────────
CURRENT_SEED = 42    # Change to 123, then 456 for subsequent runs
# ───────────────────────────────────────────────────────────────
print(f'Seed: {CURRENT_SEED}')

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/skin_lesion_v3/'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Drive mounted. Saving to: {SAVE_DIR}')

In [ ]:
!pip install -q kaggle timm scikit-learn matplotlib seaborn pandas Pillow opencv-python-headless scipy

In [ ]:
import os, io, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.model_selection import GroupShuffleSplit
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy import stats as scipy_stats

# ── Reproducibility ─────────────────────────────────────────────
SEED = CURRENT_SEED
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 25
NUM_CLASSES= 7

CLASS_NAMES = {
    'nv':    'Melanocytic Nevi',
    'mel':   'Melanoma',
    'bkl':   'Benign Keratosis',
    'bcc':   'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratoses',
    'vasc':  'Vascular Lesions',
    'df':    'Dermatofibroma'
}
CLASS_TO_IDX = {k: i for i, k in enumerate(CLASS_NAMES.keys())}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

DEGRADATION_PARAMS = {
    'Gaussian Noise':          [0, 10, 25, 50, 75, 100],
    'Gaussian Blur':           [0, 1, 2, 3, 5, 7],
    'Resolution Downsampling': [224, 112, 56, 28, 14, 7],
    'JPEG Compression':        [100, 80, 60, 40, 20, 5],
}

# ── Model definitions ────────────────────────────────────────────
# NOTE: All ViT variants use augreg_in1k weights (ImageNet-1k pretraining)
# This ensures FAIR comparison — same pretraining dataset across all ViT sizes
MODEL_BUILDERS = {
    'EfficientNetB0 (CNN, 4M)':       lambda: timm.create_model('efficientnet_b0',                    pretrained=True, num_classes=NUM_CLASSES),
    'ViT-Ti/16 (Transformer, 5.7M)':  lambda: timm.create_model('vit_tiny_patch16_224.augreg_in1k',  pretrained=True, num_classes=NUM_CLASSES),
    'ViT-S/16 (Transformer, 22M)':    lambda: timm.create_model('vit_small_patch16_224.augreg_in1k', pretrained=True, num_classes=NUM_CLASSES),
    'ViT-B/16 (Transformer, 86M)':    lambda: timm.create_model('vit_base_patch16_224.augreg_in1k',  pretrained=True, num_classes=NUM_CLASSES),
    'EfficientFormer-L1 (Hybrid, 11M)':lambda: timm.create_model('efficientformer_l1',               pretrained=True, num_classes=NUM_CLASSES),
}

print(f'Device: {DEVICE} | Seed: {SEED}')
print(f'Models: {list(MODEL_BUILDERS.keys())}')

## Step 2 — Dataset Download

In [ ]:
if not os.path.exists('data/HAM10000_metadata.csv'):
    from google.colab import files
    print('Upload your kaggle.json:')
    files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
    !unzip -q skin-cancer-mnist-ham10000.zip -d data/
    print('Done!')
else:
    print('Data already present.')

## Step 3 — Lesion-ID Aware Split

**This is the critical fix from reviewer feedback.**

HAM10000 contains ~10,015 images but only ~7,470 unique lesions.
Some lesions were photographed multiple times (different dermoscopes/angles).

**Problem with naive random split:**
The same lesion could appear in both train AND test → inflated test performance.

**Fix:**
Split by `lesion_id` groups. All images of the same lesion stay in the same split.
This is the standard practice in HAM10000 literature.

In [ ]:
df = pd.read_csv('data/HAM10000_metadata.csv')
df['label'] = df['dx'].map(CLASS_TO_IDX)

# Find image paths
img_dirs = ['data/HAM10000_images_part_1', 'data/HAM10000_images_part_2']
img_path_map = {}
for d in img_dirs:
    if os.path.exists(d):
        for f in os.listdir(d):
            if f.endswith('.jpg'):
                img_path_map[f.replace('.jpg', '')] = os.path.join(d, f)

df['path'] = df['image_id'].map(img_path_map)
df = df.dropna(subset=['path'])

# ── Report duplicate lesion situation ──────────────────────────
n_images  = len(df)
n_lesions = df['lesion_id'].nunique()
print(f'Total images:  {n_images}')
print(f'Unique lesions: {n_lesions}')
print(f'Duplicate images: {n_images - n_lesions} ({(n_images-n_lesions)/n_images*100:.1f}%)')
print()

# ── Lesion-ID aware stratified split ───────────────────────────
# Get one row per lesion (for splitting purposes)
lesion_df = df.groupby('lesion_id').first().reset_index()

# Split lesion IDs: 70% train, 15% val, 15% test
# Use GroupShuffleSplit to ensure lesion_id groups stay together
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
train_lesion_idx, temp_lesion_idx = next(gss.split(
    lesion_df, lesion_df['dx'], groups=lesion_df['lesion_id']
))

train_lesions = set(lesion_df.iloc[train_lesion_idx]['lesion_id'])
temp_lesions  = set(lesion_df.iloc[temp_lesion_idx]['lesion_id'])

# Split temp into val/test
temp_lesion_df = lesion_df[lesion_df['lesion_id'].isin(temp_lesions)]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_lesion_idx, test_lesion_idx = next(gss2.split(
    temp_lesion_df, temp_lesion_df['dx'], groups=temp_lesion_df['lesion_id']
))

val_lesions  = set(temp_lesion_df.iloc[val_lesion_idx]['lesion_id'])
test_lesions = set(temp_lesion_df.iloc[test_lesion_idx]['lesion_id'])

# Assign images to splits based on their lesion_id
train_df = df[df['lesion_id'].isin(train_lesions)].reset_index(drop=True)
val_df   = df[df['lesion_id'].isin(val_lesions)].reset_index(drop=True)
test_df  = df[df['lesion_id'].isin(test_lesions)].reset_index(drop=True)

# ── Verify NO lesion overlap between splits ─────────────────────
train_les = set(train_df['lesion_id'])
val_les   = set(val_df['lesion_id'])
test_les  = set(test_df['lesion_id'])

assert len(train_les & test_les) == 0, 'ERROR: Train/test lesion overlap!'
assert len(train_les & val_les)  == 0, 'ERROR: Train/val lesion overlap!'
assert len(val_les   & test_les) == 0, 'ERROR: Val/test lesion overlap!'

print(f'Train: {len(train_df)} images ({len(train_les)} unique lesions)')
print(f'Val:   {len(val_df)} images ({len(val_les)} unique lesions)')
print(f'Test:  {len(test_df)} images ({len(test_les)} unique lesions)')
print()
print('✓ Zero lesion overlap between splits — split is correct')

# Save test_df to Drive
test_df.to_csv(f'{SAVE_DIR}test_df_seed{SEED}.csv', index=False)
print(f'Test split saved to Drive.')

# Class weights
class_counts  = train_df['label'].value_counts().sort_index().values
class_weights = torch.FloatTensor(1.0 / class_counts)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights = class_weights.to(DEVICE)

## Step 4 — Degradation Functions

In [ ]:
def apply_gaussian_noise(img_tensor, std):
    if std == 0: return img_tensor
    return torch.clamp(img_tensor + torch.randn_like(img_tensor) * (std / 255.0), 0, 1)

def apply_gaussian_blur(pil_img, radius):
    if radius == 0: return pil_img
    return pil_img.filter(ImageFilter.GaussianBlur(radius=radius))

def apply_downsampling(pil_img, target_size):
    if target_size == IMG_SIZE: return pil_img
    return pil_img.resize((target_size, target_size), Image.BILINEAR).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

def apply_jpeg_compression(pil_img, quality):
    if quality == 100: return pil_img
    buf = io.BytesIO()
    pil_img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')

print('Degradation functions ready.')

## Step 5 — Dataset & DataLoaders

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None, degradation_type=None, severity=0):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.degradation_type = degradation_type
        self.severity = severity

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))

        if self.degradation_type == 'Gaussian Blur' and self.severity > 0:
            img = apply_gaussian_blur(img, DEGRADATION_PARAMS['Gaussian Blur'][self.severity])
        elif self.degradation_type == 'Resolution Downsampling' and self.severity > 0:
            img = apply_downsampling(img, DEGRADATION_PARAMS['Resolution Downsampling'][self.severity])
        elif self.degradation_type == 'JPEG Compression' and self.severity > 0:
            img = apply_jpeg_compression(img, DEGRADATION_PARAMS['JPEG Compression'][self.severity])

        if self.transform: img = self.transform(img)
        else: img = transforms.ToTensor()(img)

        if self.degradation_type == 'Gaussian Noise' and self.severity > 0:
            img = apply_gaussian_noise(img, DEGRADATION_PARAMS['Gaussian Noise'][self.severity])

        return img, torch.tensor(row['label'], dtype=torch.long)


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_loader = DataLoader(SkinLesionDataset(train_df, transform=train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(SkinLesionDataset(val_df, transform=eval_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Step 6 — Training

**Training protocol (fully specified for paper methods section):**
- Optimizer: AdamW, lr=1e-4, weight_decay=1e-4
- Schedule: Cosine annealing, T_max=25, no warmup
- Epochs: 25
- Batch size: 32
- No gradient clipping
- Best weights by peak validation weighted F1
- All ViT variants: same ImageNet-1k pretraining (augreg_in1k)
- Each model saved to Drive immediately on completion

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += out.argmax(1).eq(labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        total_loss += criterion(out, labels).item() * imgs.size(0)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return (total_loss / len(loader.dataset),
            accuracy_score(all_labels, all_preds),
            f1_score(all_labels, all_preds, average='weighted'))


def safe_filename(name):
    """Convert model name to safe filename."""
    import re
    return re.sub(r'[^\w]', '_', name.split('(')[0].strip())


def train_model(name, model_fn):
    safe_name = safe_filename(name)
    ckpt_path = f'{SAVE_DIR}{safe_name}_seed{SEED}.pth'

    # Skip if already trained
    if os.path.exists(ckpt_path):
        print(f'\nSkipping {name} — checkpoint found, loading from Drive...')
        model = model_fn().to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'  Loaded!')
        return model

    print(f'\n{"="*55}\n Training: {name} | Seed: {SEED}\n{"="*55}')
    model    = model_fn().to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best_f1, best_state = 0, None

    for epoch in range(1, EPOCHS + 1):
        t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        v_loss, v_acc, v_f1 = evaluate(model, val_loader, criterion)
        scheduler.step()
        if v_f1 > best_f1:
            best_f1 = v_f1
            best_state = deepcopy(model.state_dict())
        if epoch % 5 == 0:
            print(f'  Epoch {epoch:3d}/{EPOCHS} | Train Acc: {t_acc:.4f} | Val Acc: {v_acc:.4f} | Val F1: {v_f1:.4f}')

    model.load_state_dict(best_state)
    print(f'  Best Val F1: {best_f1:.4f}')
    torch.save(model.state_dict(), ckpt_path)
    print(f'  SAVED TO DRIVE: {ckpt_path}')
    return model


# ── Train all 5 models ──────────────────────────────────────────
# Estimated time: ~5-6 hours on T4 GPU
trained_models = {}
for name, builder in MODEL_BUILDERS.items():
    trained_models[name] = train_model(name, builder)

print('\nAll models ready!')

## Step 7 — Robustness Evaluation

63 conditions per model × 5 models = 315 total evaluations.
On GPU: ~30-40 min. On CPU: ~2-3 hours.

In [ ]:
@torch.no_grad()
def evaluate_robustness(model, test_df, degradation_type=None, severity=0):
    model.eval()
    dataset = SkinLesionDataset(test_df, transform=eval_transform,
                                degradation_type=degradation_type, severity=severity)
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        out = model(imgs.to(DEVICE))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    # Per-class F1
    f1_per_class = f1_score(all_labels, all_preds, average=None, labels=list(range(NUM_CLASSES)))
    return acc, f1, f1_per_class


print('Running robustness evaluation (315 conditions)...')
rows = []
per_class_rows = []

for model_name, model in trained_models.items():
    print(f'\n  Evaluating: {model_name}')
    for deg_type in DEGRADATION_PARAMS:
        for severity in range(6):
            acc, f1, f1_classes = evaluate_robustness(
                model, test_df,
                degradation_type=deg_type if severity > 0 else None,
                severity=severity
            )
            rows.append({
                'Seed': SEED, 'Model': model_name,
                'Degradation': deg_type, 'Severity': severity,
                'Accuracy': round(acc, 4), 'F1_Weighted': round(f1, 4),
            })
            # Per-class results
            for class_idx, class_f1 in enumerate(f1_classes):
                class_code = IDX_TO_CLASS[class_idx]
                class_name = CLASS_NAMES[class_code]
                per_class_rows.append({
                    'Seed': SEED, 'Model': model_name,
                    'Degradation': deg_type, 'Severity': severity,
                    'Class_Code': class_code, 'Class_Name': class_name,
                    'F1': round(class_f1, 4)
                })

# Save both result files
seed_df = pd.DataFrame(rows)
per_class_df = pd.DataFrame(per_class_rows)

for df_out, fname in [
    (seed_df,       f'robustness_seed{SEED}.csv'),
    (per_class_df,  f'robustness_perclass_seed{SEED}.csv')
]:
    df_out.to_csv(fname, index=False)
    df_out.to_csv(f'{SAVE_DIR}{fname}', index=False)

print(f'\nSaved: robustness_seed{SEED}.csv ({len(seed_df)} rows)')
print(f'Saved: robustness_perclass_seed{SEED}.csv ({len(per_class_df)} rows)')
print('\nMean F1 by model (clean baseline):')
clean = seed_df[seed_df['Severity']==0]
print(clean.groupby('Model')['F1_Weighted'].mean().round(4).to_string())

## Step 8 — Download This Seed's Results

In [ ]:
from google.colab import files
import shutil

os.makedirs(f'seed{SEED}_results', exist_ok=True)
shutil.copy(f'robustness_seed{SEED}.csv',          f'seed{SEED}_results/')
shutil.copy(f'robustness_perclass_seed{SEED}.csv', f'seed{SEED}_results/')
shutil.make_archive(f'seed{SEED}_results', 'zip', f'seed{SEED}_results')
files.download(f'seed{SEED}_results.zip')
print(f'Downloaded seed{SEED}_results.zip')
print(f'Also saved to Drive: {SAVE_DIR}')
print()
if SEED == 42:  print('Next: reopen notebook, set CURRENT_SEED = 123, run again.')
elif SEED == 123: print('Next: reopen notebook, set CURRENT_SEED = 456, run again.')
elif SEED == 456: print('All seeds done! Run Part 2 below to combine everything.')

---
# PART 2 — Combine All 3 Seeds & Generate Final Figures

**Run after you have all 3 seeds complete.**

Files needed in Drive (`skin_lesion_v3/`):
- `robustness_seed42.csv`, `robustness_seed123.csv`, `robustness_seed456.csv`
- `robustness_perclass_seed42.csv`, `robustness_perclass_seed123.csv`, `robustness_perclass_seed456.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats
import os

from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/skin_lesion_v3/'

# Load all 3 seeds
dfs, per_class_dfs = [], []
for seed in [42, 123, 456]:
    for dfs_list, fname_template in [
        (dfs,           f'robustness_seed{seed}.csv'),
        (per_class_dfs, f'robustness_perclass_seed{seed}.csv')
    ]:
        path = f'{SAVE_DIR}{fname_template}'
        if os.path.exists(path):
            tmp = pd.read_csv(path)
            if 'Seed' not in tmp.columns: tmp['Seed'] = seed
            dfs_list.append(tmp)
            print(f'Loaded {fname_template}: {len(tmp)} rows')
        else:
            print(f'WARNING: {fname_template} not found')

combined    = pd.concat(dfs, ignore_index=True)
combined_pc = pd.concat(per_class_dfs, ignore_index=True)
print(f'\nTotal robustness rows: {len(combined)}')
print(f'Total per-class rows:  {len(combined_pc)}')
print(f'Seeds: {sorted(combined["Seed"].unique())}')
print(f'Models: {combined["Model"].unique().tolist()}')

In [ ]:
# ── Mean ± std across seeds ─────────────────────────────────────
stats = combined.groupby(['Model', 'Degradation', 'Severity']).agg(
    Accuracy_Mean=('Accuracy', 'mean'),
    Accuracy_Std=('Accuracy', 'std'),
    F1_Mean=('F1_Weighted', 'mean'),
    F1_Std=('F1_Weighted', 'std'),
).reset_index().round(4)

stats.to_csv('robustness_stats_final_v3.csv', index=False)
stats.to_csv(f'{SAVE_DIR}robustness_stats_final_v3.csv', index=False)
print('Saved: robustness_stats_final_v3.csv')

# F1 drop summary
print('\nF1 DROP SUMMARY (mean ± std, 3 seeds)')
print('='*80)
drop_rows = []
for model in stats['Model'].unique():
    for deg in stats['Degradation'].unique():
        sub = stats[(stats['Model']==model) & (stats['Degradation']==deg)]
        clean      = sub[sub['Severity']==0]['F1_Mean'].values[0]
        worst_mean = sub[sub['Severity']==5]['F1_Mean'].values[0]
        worst_std  = sub[sub['Severity']==5]['F1_Std'].values[0]
        drop = (clean - worst_mean) * 100
        print(f'{model[:30]:30s} | {deg[:25]:25s} | {clean:.4f} -> {worst_mean:.4f}±{worst_std:.4f} | drop={drop:.1f}pp')
        drop_rows.append({'Model': model, 'Degradation': deg,
                          'Clean_F1': clean, 'Worst_F1_Mean': worst_mean,
                          'Worst_F1_Std': worst_std, 'F1_Drop_pp': round(drop,1)})

drop_df = pd.DataFrame(drop_rows)
drop_df.to_csv('f1_drop_summary_v3.csv', index=False)
drop_df.to_csv(f'{SAVE_DIR}f1_drop_summary_v3.csv', index=False)
print('\nSaved: f1_drop_summary_v3.csv')

In [ ]:
# ── Robustness curves with error bands ─────────────────────────
MODEL_COLORS = {
    'EfficientNetB0 (CNN, 4M)':        '#e74c3c',
    'ViT-Ti/16 (Transformer, 5.7M)':   '#9b59b6',
    'ViT-S/16 (Transformer, 22M)':     '#3498db',
    'ViT-B/16 (Transformer, 86M)':     '#1abc9c',
    'EfficientFormer-L1 (Hybrid, 11M)':'#2ecc71',
}
MODEL_MARKERS = {
    'EfficientNetB0 (CNN, 4M)':        'o',
    'ViT-Ti/16 (Transformer, 5.7M)':   'v',
    'ViT-S/16 (Transformer, 22M)':     's',
    'ViT-B/16 (Transformer, 86M)':     'D',
    'EfficientFormer-L1 (Hybrid, 11M)':'^',
}
DEG_LABELS = ['Gaussian Noise', 'Gaussian Blur', 'Resolution Downsampling', 'JPEG Compression']

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle(
    'Robustness to Image Degradation — Weighted F1 Score\n'
    '(Mean ± Std across 3 seeds; lesion-ID-aware split; matched-scale ViT comparison)',
    fontsize=13, fontweight='bold'
)
axes = axes.flatten()

for idx, deg_type in enumerate(DEG_LABELS):
    ax = axes[idx]
    for model_name in MODEL_COLORS:
        sub = stats[(stats['Model']==model_name) & (stats['Degradation']==deg_type)].sort_values('Severity')
        if len(sub) == 0: continue
        sev   = sub['Severity'].values
        means = sub['F1_Mean'].values
        stds  = sub['F1_Std'].values
        color  = MODEL_COLORS[model_name]
        marker = MODEL_MARKERS[model_name]
        ax.plot(sev, means, color=color, marker=marker,
                label=model_name, linewidth=2, markersize=6)
        ax.fill_between(sev, means-stds, means+stds, alpha=0.12, color=color)
    ax.set_title(deg_type, fontweight='bold', fontsize=11)
    ax.set_xlabel('Severity Level (0=Clean, 5=Worst)')
    ax.set_ylabel('Weighted F1 Score')
    ax.set_xticks(range(6))
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('robustness_curves_v3.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}robustness_curves_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: robustness_curves_v3.png')

In [ ]:
# ── Per-class robustness for ALL 4 degradation types ────────────
# This addresses reviewer feedback: per-class analysis for all degradation types

pc_stats = combined_pc.groupby(
    ['Model', 'Degradation', 'Severity', 'Class_Code', 'Class_Name']
).agg(
    F1_Mean=('F1', 'mean'),
    F1_Std=('F1', 'std')
).reset_index().round(4)

# F1 drop per class: clean vs worst severity
pc_drop_rows = []
for model in pc_stats['Model'].unique():
    for deg in DEG_LABELS:
        for class_code in pc_stats['Class_Code'].unique():
            sub = pc_stats[
                (pc_stats['Model']==model) &
                (pc_stats['Degradation']==deg) &
                (pc_stats['Class_Code']==class_code)
            ]
            if len(sub) == 0: continue
            clean = sub[sub['Severity']==0]['F1_Mean'].values
            worst = sub[sub['Severity']==5]['F1_Mean'].values
            if len(clean)==0 or len(worst)==0: continue
            class_name = sub['Class_Name'].values[0]
            drop = (clean[0] - worst[0]) * 100
            pc_drop_rows.append({
                'Model': model, 'Degradation': deg,
                'Class_Code': class_code, 'Class_Name': class_name,
                'Clean_F1': round(clean[0],4), 'Worst_F1': round(worst[0],4),
                'F1_Drop_pp': round(drop,1)
            })

pc_drop_df = pd.DataFrame(pc_drop_rows)
pc_drop_df.to_csv('perclass_drop_v3.csv', index=False)
pc_drop_df.to_csv(f'{SAVE_DIR}perclass_drop_v3.csv', index=False)
print('Saved: perclass_drop_v3.csv')

# ── Plot per-class drops for all 4 degradation types ────────────
# Focus on EfficientNetB0 and ViT-B/16 for clarity (most contrasting)
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(
    'Per-Class F1 Drop (Clean → Worst Severity) Across All Degradation Types\n'
    '(Mean across 3 seeds; higher = less robust)',
    fontsize=13, fontweight='bold'
)
axes = axes.flatten()

PLOT_MODELS = [
    'EfficientNetB0 (CNN, 4M)',
    'ViT-Ti/16 (Transformer, 5.7M)',
    'ViT-B/16 (Transformer, 86M)',
    'EfficientFormer-L1 (Hybrid, 11M)'
]
MODEL_SHORT = {
    'EfficientNetB0 (CNN, 4M)':        'CNN (4M)',
    'ViT-Ti/16 (Transformer, 5.7M)':   'ViT-Ti (5.7M)',
    'ViT-B/16 (Transformer, 86M)':     'ViT-B (86M)',
    'EfficientFormer-L1 (Hybrid, 11M)':'Hybrid (11M)'
}
PLOT_COLORS = ['#e74c3c', '#9b59b6', '#1abc9c', '#2ecc71']

CLASS_ORDER = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
CLASS_LABELS = [CLASS_NAMES[c] for c in CLASS_ORDER]

for idx, deg_type in enumerate(DEG_LABELS):
    ax = axes[idx]
    x = np.arange(len(CLASS_ORDER))
    width = 0.18

    for mi, model_name in enumerate(PLOT_MODELS):
        sub = pc_drop_df[
            (pc_drop_df['Model']==model_name) &
            (pc_drop_df['Degradation']==deg_type)
        ].set_index('Class_Code')
        drops = [sub.loc[c, 'F1_Drop_pp'] if c in sub.index else 0 for c in CLASS_ORDER]
        offset = (mi - 1.5) * width
        ax.bar(x + offset, drops, width, label=MODEL_SHORT[model_name],
               color=PLOT_COLORS[mi], alpha=0.85, edgecolor='black', linewidth=0.5)

    ax.set_title(deg_type, fontweight='bold', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_LABELS, rotation=25, ha='right', fontsize=8)
    ax.set_ylabel('F1 Drop (pp)')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('perclass_robustness_v3.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}perclass_robustness_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: perclass_robustness_v3.png')

In [ ]:
# ── Statistical significance: ViT scaling curve ─────────────────
# Tests whether ViT robustness advantage holds at MATCHED scale (ViT-Ti vs CNN)
# This directly addresses the parameter count confound

print('Statistical Significance Tests')
print('='*70)
print()
print('1. ViT-Ti (5.7M) vs CNN (4.0M) — size-matched comparison')
print('-'*70)

sig_rows = []
for deg_type in DEG_LABELS:
    cnn   = combined[(combined['Model']=='EfficientNetB0 (CNN, 4M)') &
                     (combined['Degradation']==deg_type) &
                     (combined['Severity']==5)]['F1_Weighted'].values
    vit_ti = combined[(combined['Model']=='ViT-Ti/16 (Transformer, 5.7M)') &
                      (combined['Degradation']==deg_type) &
                      (combined['Severity']==5)]['F1_Weighted'].values
    vit_s  = combined[(combined['Model']=='ViT-S/16 (Transformer, 22M)') &
                      (combined['Degradation']==deg_type) &
                      (combined['Severity']==5)]['F1_Weighted'].values
    vit_b  = combined[(combined['Model']=='ViT-B/16 (Transformer, 86M)') &
                      (combined['Degradation']==deg_type) &
                      (combined['Severity']==5)]['F1_Weighted'].values

    for model_vals, model_label in [(vit_ti, 'ViT-Ti'), (vit_s, 'ViT-S'), (vit_b, 'ViT-B')]:
        if len(cnn)>=3 and len(model_vals)>=3:
            _, p = scipy_stats.wilcoxon(model_vals, cnn, alternative='greater')
            sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
            print(f'{deg_type:25s} | {model_label} vs CNN | {model_label}={np.mean(model_vals):.4f} CNN={np.mean(cnn):.4f} | p={p:.4f} {sig}')
            sig_rows.append({
                'Degradation': deg_type, 'Comparison': f'{model_label} vs CNN',
                'Model_F1': round(np.mean(model_vals),4), 'CNN_F1': round(np.mean(cnn),4),
                'p_value': round(p,4), 'significance': sig
            })

sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv('significance_tests_v3.csv', index=False)
sig_df.to_csv(f'{SAVE_DIR}significance_tests_v3.csv', index=False)
print('\nSaved: significance_tests_v3.csv')
print('* p<0.05  ** p<0.01  *** p<0.001  ns=not significant')
print('Note: minimum p with n=3 is 0.125 — directional consistency is primary evidence')

In [ ]:
# ── ViT scaling analysis ─────────────────────────────────────────
# Does robustness scale with model size within the ViT family?
# This is a secondary finding that adds depth to the paper

print('ViT Scaling Analysis: Does robustness scale with parameter count?')
print('='*65)

vit_models = [
    ('ViT-Ti/16 (Transformer, 5.7M)',  5.7),
    ('ViT-S/16 (Transformer, 22M)',   22.0),
    ('ViT-B/16 (Transformer, 86M)',   86.0),
]

for deg_type in DEG_LABELS:
    print(f'\n{deg_type}:')
    for model_name, params in vit_models:
        sub = drop_df[(drop_df['Model']==model_name) & (drop_df['Degradation']==deg_type)]
        if len(sub) > 0:
            drop = sub['F1_Drop_pp'].values[0]
            worst = sub['Worst_F1_Mean'].values[0]
            print(f'  {params:5.1f}M params → worst F1={worst:.4f}, drop={drop:.1f}pp')

In [ ]:
# ── Download final package ──────────────────────────────────────
import shutil
from google.colab import files

os.makedirs('final_results_v3', exist_ok=True)
for f in [
    'robustness_stats_final_v3.csv',
    'f1_drop_summary_v3.csv',
    'significance_tests_v3.csv',
    'perclass_drop_v3.csv',
    'robustness_curves_v3.png',
    'perclass_robustness_v3.png',
]:
    if os.path.exists(f):
        shutil.copy(f, f'final_results_v3/{f}')

shutil.make_archive('final_results_v3', 'zip', 'final_results_v3')
files.download('final_results_v3.zip')
print('Downloaded final_results_v3.zip')
print(f'Everything saved to Drive: {SAVE_DIR}')
print()
print('Files inside:')
print('  robustness_stats_final_v3.csv   ← mean ± std for all conditions')
print('  f1_drop_summary_v3.csv          ← F1 drop table')
print('  significance_tests_v3.csv       ← p-values for all ViT vs CNN comparisons')
print('  perclass_drop_v3.csv            ← per-class drops for all 4 degradation types')
print('  robustness_curves_v3.png        ← main figure with error bands (5 models)')
print('  perclass_robustness_v3.png      ← per-class figure for all 4 degradation types')